In [1]:
# ===== Cell 1: 固定 GPU（必须最先运行） =====
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
print("当前 notebook 只会看到 1 块 GPU，并且对应物理 GPU 0")

CUDA_VISIBLE_DEVICES = 0
当前 notebook 只会看到 1 块 GPU，并且对应物理 GPU 0


In [2]:
# ===== Cell 2: 导入依赖 + 确认导入的是你本地源码 =====
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch

warnings.filterwarnings("ignore")

REPO_ROOT = "/home/zhangjunyi/xiangmu/nichecompass-main"
LOCAL_SRC = f"{REPO_ROOT}/src"

if LOCAL_SRC not in sys.path:
    sys.path.insert(0, LOCAL_SRC)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import nichecompass as nc

from analysis.data_analysis.tmcn_cluster_pipeline import (
    DEFAULT_PATHWAY_GENE_SETS,
    assign_cluster_labels,
    assign_spot_level_flags,
    compute_axis_scores,
    compute_pathology_evaluation,
    ensure_latent_clusters,
    ensure_spatial_graph,
    load_axis_definitions,
    refine_to_connected_regions,
)

print("nichecompass loaded from:")
print(nc.__file__)
print("torch.cuda.is_available() =", torch.cuda.is_available())
print("torch.cuda.device_count() =", torch.cuda.device_count())
if torch.cuda.is_available():
    print("当前 GPU =", torch.cuda.get_device_name(0))

assert nc.__file__.startswith(f"{REPO_ROOT}/src"), "当前导入的不是你本地修改后的 NicheCompass，请先重启 kernel 再运行。"

ModuleNotFoundError: No module named 'analysis.data_analysis.tmcn_cluster_pipeline'

In [ ]:
# ===== Cell 3: 配置区（你重点修改这里） =====
DATA_PATH = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_integrated.h5ad"
QUADRUPLET_CSV = "/home/zhangjunyi/xiangmu/nichecompass-main/data/pre_data/siyuanzu/my_metabolite_network_simplify.csv"
OUTPUT_DIR = "/home/zhangjunyi/xiangmu/nichecompass-main/artifacts/ViHBC_TMCN_run"
PATHOLOGY_KEY = "annot_type"   # 如果你的病理标签列名不是 annot_type，就改这里

SPATIAL_KEY = "spatial"
SPATIAL_CONNECTIVITIES_KEY = "spatial_connectivities"
LATENT_KEY = "tmcn_latent"
CLUSTER_KEY = "TMCN_Clusters"

# 训练参数：第一次建议先用小一点，跑通后再调大
N_TOP_HVG = 2000
N_SPATIAL_NEIGHBORS = 8
CLUSTER_RESOLUTION = 0.60
SPOT_HIGH_QUANTILE = 0.80
MIN_REGION_SIZE = 20

N_EPOCHS = 100
N_EPOCHS_ALL_GPS = 20
EDGE_BATCH_SIZE = 256
NODE_BATCH_SIZE = 512
USE_CUDA_IF_AVAILABLE = True

# 如果你想手动修改通路 marker，就复制 DEFAULT_PATHWAY_GENE_SETS 后在这里改
PATHWAY_GENE_SETS = DEFAULT_PATHWAY_GENE_SETS.copy()

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR =", OUTPUT_DIR)

In [ ]:
# ===== Cell 4: 读取四元组 =====
axes = load_axis_definitions(QUADRUPLET_CSV)
axis_names = [axis.axis_name for axis in axes]

df_axes = pd.DataFrame([
    {
        "axis_name": axis.axis_name,
        "source_pathways": ", ".join(axis.pathway_names),
        "source_genes": ", ".join(axis.source_genes),
        "target_genes": ", ".join(axis.target_genes),
        "meaning": axis.meaning,
    }
    for axis in axes
])

print("共读取四元组条数：", len(axes))
display(df_axes)